In [1]:
import pandas as p
from django.db.models import Q

In [2]:
strains = {s.boonelab_id: s for s in TriStrain.objects.all()}

In [6]:
df = p.read_table('/home/matej/ek_scores.txt')
df.columns = ['queryid', 'arrayid', 'score', 'pvalue']

In [8]:
df['query'] = df.queryid.apply(lambda x: x.split('_')[-1])
df['array'] = df.arrayid.apply(lambda x: x.split('_')[-1])

In [15]:
for _,_,s,p,q,a in df.itertuples(index=False):
    TriScores.objects.create(
        query=strains[q],
        array=strains[a],
        score=s,
        pvalue=p
    )

In [19]:
triplets = p.read_excel('/home/matej/AdditionalDataTableS6.xlsx')
for d,s1,s2 in triplets.iloc[:,:3].itertuples(index=False):
    TriStrainSet.objects.create(
        double_mutant=strains[d],
        single_mutant1=strains[s1],
        single_mutant2=strains[s2]
    )

In [2]:
strainMap = {}
for s in TriStrain.objects.filter(
    Q(double_in_set__isnull=False) | 
    Q(single1_in_set__isnull=False) | 
    Q(single2_in_set__isnull=False) |
    Q(is_query=False)
).distinct():
    strainMap[s.pk] = s.to_dict()

In [4]:
import json

In [5]:
with open('./trigenic/static/files/strains.json', 'w') as fout:
    fout.write(json.dumps(strainMap).replace(' ', ''))